# Offline outcome table: review every context level

Builds the (PR × level) outcome table described in roadmap Section 8.5. The frozen reviewer reviews each level's `context.md`, AACR-Bench's judge scores each review against the reference comments, and the results go to `data/aacr_tracer/outcomes/<run_name>/`.

- **Reviewer prompt:** OpenAI Codex's built-in `/review` rubric, verbatim, as the system message (`configs/review_prompts/codex_review_rubric.md`, from openai/codex at commit `4e4767f`, Apache-2.0). The user message is a one-sentence request followed by the level's `context.md`. Codex's own request tells the reviewer to run `git diff`; a one-call reviewer cannot, so only that sentence is adapted.
- **Reviewer model:** `Qwen/Qwen3.8-27B`, greedy decoding, served by vLLM on the cloud GPU (profile `cloud`).
  - A laptop cannot run it.
  - Profile `local-smoke` runs `Qwen/Qwen3.5-0.8B` in-process on two short levels, only to test the pipeline. Its outputs are **not results**.
- **Judge:** AACR-Bench's file-level semantic judge, with its prompt copied verbatim. **The judge model is still an open decision** (roadmap Section 8.5.3). `keyword-test` is a stand-in for pipeline tests only.
- **Inputs:** the context levels written by `scripts/run_tracer_batch.py` (`data/aacr_tracer/entries/<id>/<level>/context.md`). Reference comments are read only by the judging stage and never reach the reviewer.
- **Resuming:** re-running reuses every cell whose input, prompt, model and settings are unchanged. Failed calls are retried.

**Choose the profile** with the `OUTCOME_PROFILE` environment variable (`cloud` by default), or edit `PROFILE` below.

## Before running the `cloud` profile

Running it uses the cloud GPU, so get the go-ahead first.

1. **Start the reviewer server** on the GPU machine. For example: `vllm serve Qwen/Qwen3.8-27B --revision 37561816e66b5b9153b1b64c10968c18c1e6f4a0 --max-model-len <N>`, with `N` covering the largest level plus about 4k output tokens. vllm#17425's `L10_all` is about 133k input tokens. Check the flags against your vLLM version's documentation.
2. **Make the inputs available on that machine:** copy `data/aacr_tracer/entries/` and AACR's `positive_samples.json`, or run `scripts/run_tracer_batch.py` there.
3. **Set `base_url` below.**
4. **Choose and serve the judge model**, then set the `judge` entry. It is still undecided.

In [ ]:
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "review_outcomes").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from tqdm.auto import tqdm

from src.review_outcomes import backends, table
from src.review_outcomes.parse import PARSER_VERSION
from src.review_outcomes.prompts import (
    JUDGE_PROMPT_VERSION, REVIEW_PROMPT_VERSION, REVIEW_REQUEST, RUBRIC_SHA256, load_rubric, prompt_sha256,
)
from src.tracer_context.aacr import file_sha256

print(PROJECT_ROOT)

## Configuration

In [ ]:
PROFILE = os.environ.get("OUTCOME_PROFILE", "cloud")  # "cloud" or "local-smoke"

WORK = PROJECT_ROOT / "data" / "aacr_tracer"
DATASET = PROJECT_ROOT / "data" / "codebase_memory_trial" / "aacr" / "positive_samples.json"

PROFILES = {
    "cloud": {
        "run_name": "qwen3.8-27b_codex-rubric_v1",
        "entries": None,  # None = every entry the batch runner finished
        "levels": None,   # None = all ten levels
        "reviewer": {"backend": "openai", "base_url": "http://localhost:8000/v1", "model": "Qwen/Qwen3.8-27B",
                     "model_revision": "37561816e66b5b9153b1b64c10968c18c1e6f4a0", "max_tokens": 4096,
                     "enable_thinking": False, "concurrency": 8, "max_input_tokens": 200_000,
                     "api_key_env": "REVIEWER_API_KEY"},
        "judge": {"backend": "openai", "base_url": "http://localhost:8001/v1", "model": "JUDGE_MODEL_TO_BE_DECIDED",
                  "max_tokens": 3000, "enable_thinking": False, "api_key_env": "JUDGE_API_KEY"},
    },
    "local-smoke": {
        "run_name": "smoke_qwen3.5-0.8b",
        "entries": ["vllm-project__vllm__17425"],
        "levels": ["L01_min", "L02_function"],
        "reviewer": {"backend": "transformers", "model": "Qwen/Qwen3.5-0.8B", "max_tokens": 1024,
                     "enable_thinking": False, "concurrency": 1, "max_input_tokens": 12_000},
        "judge": {"backend": "keyword-test"},
    },
}
CONFIG = PROFILES[PROFILE]
OUT = WORK / "outcomes" / CONFIG["run_name"]
print(PROFILE, "->", OUT)

In [ ]:
def make_chat(spec):
    if spec["backend"] == "openai":
        return backends.OpenAIChat(spec["base_url"], spec["model"], api_key=os.environ.get(spec.get("api_key_env", "")),
                                   max_tokens=spec["max_tokens"], enable_thinking=spec.get("enable_thinking"),
                                   model_revision=spec.get("model_revision"))
    if spec["backend"] == "transformers":
        return backends.TransformersChat(spec["model"], max_new_tokens=spec["max_tokens"],
                                         enable_thinking=spec.get("enable_thinking"))
    raise ValueError(f"unknown backend {spec['backend']!r}")


rubric = load_rubric()  # refuses to run if the pinned rubric file was edited
reviewer = make_chat(CONFIG["reviewer"])
print("reviewer:", reviewer.identity)
print("prompt sha256:", prompt_sha256(rubric))

## 1. Find the cells

One cell per (entry, level) with a `context.md`. Cells above `max_input_tokens` are recorded as `too_long` instead of being sent.

In [ ]:
cells = table.discover_cells(WORK / "entries", CONFIG["entries"], CONFIG["levels"])
cells_df = pd.DataFrame([{"entry": c.entry, "level": c.level, "input_tokens": c.input_tokens} for c in cells])
limit = CONFIG["reviewer"]["max_input_tokens"]
print(f"{len(cells)} cells from {cells_df.entry.nunique()} entries; "
      f"{int((cells_df.input_tokens > limit).sum())} over {limit} input tokens")
cells_df.groupby("level").input_tokens.agg(["count", "mean", "max"])

## 2. Review

Saved per cell in `reviews/<entry>/<level>.json`: the raw reply, the parsed findings, token counts, the prompt hash and the backend settings. Re-run this cell to resume or to retry errors.

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
git_head = subprocess.run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
table.write_json(OUT / "manifest.json", {
    "profile": PROFILE, "config": CONFIG, "written_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "code_commit": git_head, "rubric_sha256": RUBRIC_SHA256, "review_request": REVIEW_REQUEST,
    "prompt_sha256": prompt_sha256(rubric), "review_prompt_version": REVIEW_PROMPT_VERSION,
    "judge_prompt_version": JUDGE_PROMPT_VERSION, "parser_version": PARSER_VERSION,
    "reviewer_identity": reviewer.identity, "dataset": str(DATASET), "dataset_sha256": file_sha256(DATASET),
})

bar = tqdm(total=len(cells), desc="reviews")
reviews = table.run_reviews(cells, reviewer, rubric, OUT, max_input_tokens=limit,
                            concurrency=CONFIG["reviewer"]["concurrency"], progress=lambda record: bar.update())
bar.close()
pd.Series([r["status"] for r in reviews]).value_counts()

In [ ]:
# Look at one reply and its parsed findings.
sample = next((r for r in reviews if r.get("reply")), None)
if sample:
    reply = sample["reply"]
    print(sample["entry"], sample["level"], sample["status"], f"{reply['completion_tokens']} output tokens,",
          f"{reply['seconds']}s, finish={reply['finish_reason']}")
    print(reply["text"][:3000])
    print(json.dumps(sample["parsed"]["findings"][:3], indent=1))

## 3. Judge

AACR's file-level semantic metric: one judge call per generated comment against that file's reference comments. Malformed replies count as empty reviews. A failed judge call is recorded and retried on the next run.

**The judge model is still open** (roadmap Section 8.5.3). `keyword-test` scores are not meaningful.

In [ ]:
spec = CONFIG["judge"]
if spec["backend"] == "keyword-test":
    judge = backends.KeywordOverlapJudge()
    print("WARNING: keyword-test judge (pipeline test only); scores are not meaningful")
else:
    if "TO_BE_DECIDED" in spec.get("model", ""):
        raise RuntimeError("Choose the judge model first (roadmap Section 8.5.3) and set CONFIG['judge'].")
    judge = backends.judge_caller(make_chat(spec))

references = table.references_by_entry(DATASET)
scorable = sum(r["status"] in ("ok", "malformed", "empty_output") for r in reviews)
bar = tqdm(total=scorable, desc="judging")
judgments = table.run_judgments(reviews, references, judge, OUT, progress=lambda record: bar.update())
bar.close()
print(sum(j["judge_errors"] for j in judgments), "judge errors (re-run this cell to retry them)")

## 4. Outcome table

`table.csv` has one row per (entry, level), and `coverage.csv` one row per reference comment per level. The summary pools counts over PRs, as AACR does, and also shows the per-PR average recall and human-only recall.

In [ ]:
rows, coverage = table.build_table(reviews, judgments, references)
table.write_outputs(OUT, rows, coverage)
print("wrote", OUT / "table.csv", "and", OUT / "coverage.csv")
pd.DataFrame(table.summarize_by_level(rows))

In [ ]:
pd.DataFrame(rows).head(20)